### AlphaGenome in-Silico Mutagenesis (ISM) Analysis — All Introns of a Gene

This notebook analyzes AlphaGenome variant scoring results from saturation mutagenesis
across **every intron of a gene** produced by an `alphagenome_ISM_<GENE>/` run.
It is gene-agnostic: point `INPUT_ROOT` at the run folder, set `OUTPUT_DIR`, and set
`GENE_NAME` (used for plot titles/labels) in the configuration cell below.

For each `intron_*` folder it:
1. Loads the score data, supporting either many per-variant `csv/alphagenome_scores_*.csv` files **or** a single aggregated CSV (e.g. `csv/alphagenome_scores_intron_N.csv`) holding all variants for the intron. In-progress `.csv.tmp` files are skipped, since they mean the intron is still being analyzed
2. Applies the same pathogenicity cutoffs and annotation as the intron-6 workflow
3. Generates per-intron ISM heatmaps and spatial hotspot plots
4. Builds a cohort-level summary to compare hotspot signal across introns

### Setup
1. Set `INPUT_ROOT` to the parent directory containing `intron_*` folders
2. Set `OUTPUT_DIR` for saving results
3. Set `GENE_NAME` for plot titles/labels
4. Optionally restrict analysis with `INTRON_FILTER`
5. Run all cells in order

## 1. Configuration

In [1]:
# ============================================
# CONFIGURATION
# ============================================

# Gene under analysis. Drives the default input/output paths and every
# plot title/label, so switching genes is usually a one-line change here.
GENE_NAME = "COL4A5"

# INPUT_ROOT: parent directory with one subfolder per intron (each with csv/)
INPUT_ROOT = f"../01_alphagenome_analysis/alphagenome_ISM_{GENE_NAME}/"
OUTPUT_DIR = f"./plots_alphagenome_ISM_{GENE_NAME}_all_introns/"

# Restrict to specific introns, e.g. ["intron_6", "intron_47"]; None = all discovered
INTRON_FILTER = None

# Pathogenicity thresholds (same as intron-6 notebook)
THRESHOLD_HIGH = 0.999      # Top 0.1%
THRESHOLD_MODERATE = 0.99   # Top 1%
THRESHOLD_LOW = 0.95        # Top 5%

# Hotspot sliding-window parameters (same as intron-6 notebook)
HOTSPOT_WINDOW = 15
HOTSPOT_STEP = 1
HOTSPOT_FDR = 0.05

# Exclude this many bp from each intron end when reporting interior H/M burden
INTRON_END_TRIM_NT = 100

# Number of top variants to display in tables
TOP_N_VARIANTS = 20

# Set False to skip per-intron heatmaps for very long introns (faster batch run)
GENERATE_ISM_HEATMAP = True
GENERATE_SCORES_HEATMAP = True

## 2. Import Libraries

In [2]:
import glob
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input root: {INPUT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Input root: ../01_alphagenome_analysis/alphagenome_ISM_COL4A5/
Output directory: ./plots_alphagenome_ISM_COL4A5_all_introns/


## 3. Data Loading Function

In [3]:
SCORE_FILE_GLOBS = ('alphagenome_scores_*.csv', '*.csv')


def find_score_files(directory) -> list:
    """Return sorted AlphaGenome score files inside a directory."""
    directory = Path(directory)
    for pattern in SCORE_FILE_GLOBS:
        # Use the first pattern that matches so we don't mix the specific
        # "alphagenome_scores_*" hits with the broad "*.csv" fallback.
        matches = sorted(glob.glob(str(directory / pattern)))
        if matches:
            return matches
    return []


def load_alphagenome_data(input_path: str) -> pd.DataFrame:
    """
    Load AlphaGenome data from any of:
      * a single per-variant CSV file,
      * a single aggregated CSV/CSV.tmp file holding many variants, or
      * a directory containing either many per-variant files or one
        aggregated file.

    Aggregated files carry an extra leading ``intron_name`` column that the
    per-variant files lack; it is dropped here so downstream code sees a
    consistent schema either way.
    """
    input_path = Path(input_path)
    
    if input_path.is_file():
        print(f"Loading single file: {input_path.name}")
        combined_df = pd.read_csv(input_path)
    elif input_path.is_dir():
        files = find_score_files(input_path)
        
        if not files:
            raise FileNotFoundError(f"No CSV files found in {input_path}")
        
        if len(files) == 1:
            print(f"Found 1 aggregated CSV file: {Path(files[0]).name}")
            combined_df = pd.read_csv(files[0])
        else:
            print(f"Found {len(files)} CSV files")
            print(f"Loading files (this may take a moment for large datasets)...")
            
            # Efficient loading with progress indication
            dfs = []
            for i, f in enumerate(files):
                if (i + 1) % 100 == 0:
                    print(f"  Loaded {i + 1}/{len(files)} files...")
                dfs.append(pd.read_csv(f))
            
            combined_df = pd.concat(dfs, ignore_index=True)
            print(f"  Loaded all {len(files)} files.")
    else:
        raise FileNotFoundError(f"Input path not found: {input_path}")
    
    # Clean up
    if 'Unnamed: 0' in combined_df.columns:
        combined_df = combined_df.drop(columns=['Unnamed: 0'])
    # Aggregated files include an extra 'intron_name' column not present in the
    # per-variant files; drop it so the schema matches downstream expectations.
    if 'intron_name' in combined_df.columns:
        combined_df = combined_df.drop(columns=['intron_name'])
    
    # Data quality checks
    print(f"\n--- Data Summary ---")
    print(f"Total rows: {combined_df.shape[0]:,}")
    print(f"Unique variants: {combined_df['variant_id'].nunique():,}")
    
    return combined_df

## 4. Helper Functions

In [4]:
def classify_pathogenicity(quantile, threshold_high=0.999, threshold_moderate=0.99, threshold_low=0.95):
    """Classify variant pathogenicity based on quantile score."""
    if quantile >= threshold_high:
        return 'High Impact'
    elif quantile >= threshold_moderate:
        return 'Moderate Impact'
    elif quantile >= threshold_low:
        return 'Low Impact'
    else:
        return 'No Impact'

def parse_variant_id(variant_id):
    """Parse variant_id to extract chromosome, position, ref, and alt."""
    parts = variant_id.split(':')
    chrom = parts[0]
    position = int(parts[1])
    change = parts[2]
    ref = change.split('>')[0]
    alt = change.split('>')[1]
    return chrom, position, ref, alt, change

## 5. Create Variant Summary

In [5]:
def create_variant_summary(combined_df, threshold_high=0.999, threshold_moderate=0.99, threshold_low=0.95):
    """
    Create summary statistics for each variant.

    Uses the same logic and output schema as the cohort notebook
    (per-output-type splice metrics + classification based on
    SPLICE_SITE_USAGE quantile), but vectorised with groupby for
    efficiency on the large ISM dataset.
    """
    # Overall stats across all output types
    overall = combined_df.groupby('variant_id').agg(
        overall_max_raw=('raw_score', 'max'),
        overall_mean_raw=('raw_score', 'mean'),
        overall_max_quantile=('quantile_score', 'max'),
        overall_mean_quantile=('quantile_score', 'mean'),
        n_predictions=('output_type', 'count'),
    ).reset_index()

    # Per-output-type stats: SPLICE_JUNCTIONS
    sj = combined_df[combined_df['output_type'] == 'SPLICE_JUNCTIONS'].groupby('variant_id').agg(
        splice_junction_max_raw=('raw_score', 'max'),
        splice_junction_mean_raw=('raw_score', 'mean'),
        splice_junction_max_quantile=('quantile_score', 'max'),
        splice_junction_mean_quantile=('quantile_score', 'mean'),
    ).reset_index()

    # Per-output-type stats: SPLICE_SITE_USAGE
    ss = combined_df[combined_df['output_type'] == 'SPLICE_SITE_USAGE'].groupby('variant_id').agg(
        splice_site_max_raw=('raw_score', 'max'),
        splice_site_mean_raw=('raw_score', 'mean'),
        splice_site_max_quantile=('quantile_score', 'max'),
        splice_site_mean_quantile=('quantile_score', 'mean'),
    ).reset_index()

    summary = overall.merge(sj, on='variant_id', how='left').merge(ss, on='variant_id', how='left')

    # Parse variant IDs into chrom/position/ref/alt/change columns
    parsed = summary['variant_id'].apply(lambda x: pd.Series(parse_variant_id(x)))
    parsed.columns = ['chrom', 'position', 'ref', 'alt', 'change']
    summary = pd.concat([summary, parsed], axis=1)

    # Short variant label (e.g. "108570657:C>G"), to match cohort notebook
    summary['short_variant'] = summary['variant_id'].apply(
        lambda x: x.split(':')[1] + ':' + x.split(':')[2]
    )

    # Classification is based on the SPLICE_SITE_USAGE quantile score only
    # (matches the cohort notebook). If a variant has no SPLICE_SITE_USAGE
    # prediction it is labelled 'Not Evaluated'.
    summary['impact_class'] = summary['splice_site_max_quantile'].apply(
        lambda x: classify_pathogenicity(x, threshold_high, threshold_moderate, threshold_low)
        if pd.notna(x) else 'Not Evaluated'
    )

    summary = summary.sort_values('overall_max_quantile', ascending=False).reset_index(drop=True)

    return summary

## 6. Visualization Functions

### ISM Heatmap (Saturation Mutagenesis Landscape)

In [6]:
def plot_ism_heatmap(summary_df, output_dir=None, top_n=None, show=True):
    """
    Create a heatmap showing SPLICE_SITE_USAGE quantile scores for each
    position and nucleotide change. This is the classic saturation
    mutagenesis visualization, and uses the same metric that drives the
    pathogenicity classification in `create_variant_summary` (cohort logic).

    Parameters
    ----------
    summary_df : pd.DataFrame
        Variant summary dataframe.
    output_dir : str, optional
        Output directory for saving plots.
    top_n : int or None, optional
        Number of highest-scoring variants to include. Use None to include all variants.
    """
    # Keep only top N variants for a focused heatmap (or all if top_n is None)
    if top_n is not None:
        plot_df = summary_df.sort_values('splice_site_max_quantile', ascending=False).head(top_n).copy()
    else:
        plot_df = summary_df.copy()

    # Create pivot table: position x alt allele, coloured by splice-site usage
    pivot_data = plot_df.pivot_table(
        index='alt',
        columns='position',
        values='splice_site_max_quantile',
        aggfunc='max'
    )

    # Reorder rows to standard nucleotide order
    nucleotide_order = ['A', 'C', 'G', 'T']
    pivot_data = pivot_data.reindex([n for n in nucleotide_order if n in pivot_data.index])
    
    # Calculate figure width based on number of positions.
    # Cap the width so that no rendering path (tight_layout, SVG bbox, or the
    # raster PNG at save dpi) exceeds matplotlib's Agg/libpng 2**16 px-per-axis
    # limit and so the font ppem stays valid. Very long introns are compressed
    # into this width (use the per-position scores CSV for nt-level detail).
    n_positions = len(pivot_data.columns)
    max_fig_width_in = 200
    fig_width = min(max_fig_width_in, max(16, n_positions * 0.12))
    
    fig, ax = plt.subplots(figsize=(fig_width, 4))
    
    # Custom colormap: green (low / no-impact scores) -> yellow -> orange -> red (pathogenic)
    colors = ['#4CAF50', '#FFC107', '#FF9800', '#D32F2F']
    cmap = LinearSegmentedColormap.from_list('pathogenicity', colors, N=256)
    
    # Create heatmap
    sns.heatmap(
        pivot_data,
        cmap=cmap,
        vmin=0.980,
        vmax=1.0,
        cbar_kws={'label': 'Splice-site usage quantile score', 'shrink': 0.8},
        ax=ax,
        xticklabels=True,
        yticklabels=True,
        linewidths=0.1,
        linecolor='white'
    )
    
    # Customize
    ax.set_xlabel('Genomic Position', fontsize=11)
    ax.set_ylabel('Alternative Allele', fontsize=11)
    ax.set_title('AlphaGenome ISM Heatmap - Saturation Mutagenesis Landscape\n(Higher score= more likely to disrupt splicing)', 
                 fontsize=12, fontweight='bold')
    
    # Rotate x-axis labels
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(fontsize=10)
    
    # Black border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.5)
    
    plt.tight_layout()
    
    if output_dir:
        # Width is capped at creation, so a 250-dpi PNG stays under the raster
        # pixel limit (<=200 in x 250 dpi = 50000 px) with a valid font ppem.
        plt.savefig(os.path.join(output_dir, 'ism_heatmap.png'), dpi=250, bbox_inches='tight')
        plt.savefig(os.path.join(output_dir, 'ism_heatmap.svg'), bbox_inches='tight')
        print("Saved: ism_heatmap.png/svg")
    
    if show:
        plt.show()
    else:
        plt.close(fig)

### Heatmap (top 20)

In [7]:
def plot_scores_heatmap(
    combined_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    output_dir: str = None,
    top_n: int = 20,
    show: bool = True,
):
    """
    Create heatmap of scores by variant and output type.

    Parameters
    ----------
    combined_df : pd.DataFrame
        Combined AlphaGenome scores dataframe
    summary_df : pd.DataFrame
        Variant summary dataframe
    output_dir : str, optional
        Output directory for saving plots
    top_n : int, optional
        Number of highest-scoring variants to include in the heatmap
    """
    # Get max quantile score for each variant and output type
    heatmap_data = combined_df.groupby(['variant_id', 'output_type'])['quantile_score'].max().reset_index()
    heatmap_data['short_variant'] = heatmap_data['variant_id'].apply(
        lambda x: x.split(':')[1] + ':' + x.split(':')[2]
    )

    # Build short_variant in summary_df if not already present
    summary_plot = summary_df.copy()
    if 'short_variant' not in summary_plot.columns:
        summary_plot['short_variant'] = summary_plot['variant_id'].apply(
            lambda x: x.split(':')[1] + ':' + x.split(':')[2]
        )

    # Sort by overall impact and keep only top N variants
    top_variants = summary_plot.sort_values('overall_max_quantile', ascending=False).head(top_n)
    order = top_variants['short_variant'].tolist()

    # Pivot for heatmap and reindex to top variants order
    heatmap_pivot = heatmap_data.pivot(index='short_variant', columns='output_type', values='quantile_score')
    heatmap_pivot = heatmap_pivot.reindex(order)

    fig, ax = plt.subplots(figsize=(6, 6))

    sns.heatmap(
        heatmap_pivot,
        annot=True,
        fmt='.4f',
        cmap='RdYlGn_r',
        vmin=0.9,
        vmax=1.0,
        linewidths=0.5,
        cbar_kws={'label': 'Quantile Score'},
        ax=ax
    )

    ax.set_title(f'AlphaGenome scores by variant and prediction type (Top {len(order)})', fontsize=10)
    ax.set_xlabel('Prediction Type', fontsize=8)
    ax.set_ylabel('', fontsize=10)

    plt.tight_layout()

    if output_dir:
        plt.savefig(os.path.join(output_dir, 'variant_scores_heatmap.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(output_dir, 'variant_scores_heatmap.svg'), bbox_inches='tight')
        print("Saved: variant_scores_heatmap.png/svg")

    if show:
        plt.show()
    else:
        plt.close(fig)

## 7. Summary Functions

In [8]:
def print_summary_statistics(summary_df):
    """
    Print comprehensive summary statistics.
    """
    print("=" * 60)
    print("ISM ANALYSIS SUMMARY")
    print("=" * 60)
    
    # Basic stats
    print(f"\nTotal variants analyzed: {len(summary_df):,}")
    print(f"Genomic region: {summary_df['position'].min():,} - {summary_df['position'].max():,}")
    print(f"Region size: {summary_df['position'].max() - summary_df['position'].min() + 1:,} bp")
    print(f"Unique positions: {summary_df['position'].nunique():,}")
    
    # Impact classification
    print("\n--- Impact Classification ---")
    for impact in ['High Impact', 'Moderate Impact', 'Low Impact', 'No Impact']:
        count = len(summary_df[summary_df['impact_class'] == impact])
        pct = count / len(summary_df) * 100
        print(f"  {impact}: {count:,} ({pct:.1f}%)")
    
    # Score statistics
    print("\n--- Score Statistics ---")
    print(f"  Max quantile score: {summary_df['overall_max_quantile'].max():.6f}")
    print(f"  Min quantile score: {summary_df['overall_max_quantile'].min():.6f}")
    print(f"  Mean quantile score: {summary_df['overall_max_quantile'].mean():.6f}")
    print(f"  Median quantile score: {summary_df['overall_max_quantile'].median():.6f}")
    
    print("\n" + "=" * 60)


def get_top_variants(summary_df, n=20):
    """
    Get top N highest-scoring variants.
    """
    top = summary_df.head(n)[['variant_id', 'position', 'change', 'impact_class',
                              'overall_max_quantile', 'splice_junction_max_quantile', 
                              'splice_site_max_quantile']].copy()
    top.columns = ['Variant', 'Position', 'Change', 'Impact', 'Max Quantile', 
                   'Splice Junction', 'Splice Site']
    return top


def save_summary_table(summary_df, output_dir):
    """
    Save the full annotated variant summary to CSV.

    All columns produced by `create_variant_summary` are written out:
        - identifiers / coordinates: variant_id, short_variant, chrom, position, ref, alt, change
        - classification: impact_class
        - overall scores (across both output types): max/mean for raw and quantile
        - per-track scores for SPLICE_JUNCTIONS and SPLICE_SITE_USAGE: max/mean for raw and quantile
        - n_predictions
    """
    # Order columns logically: ids -> coords -> classification -> overall -> per-track -> count
    column_order = [
        'variant_id', 'short_variant', 'chrom', 'position', 'ref', 'alt', 'change',
        'impact_class',
        'overall_max_raw', 'overall_mean_raw',
        'overall_max_quantile', 'overall_mean_quantile',
        'splice_junction_max_raw', 'splice_junction_mean_raw',
        'splice_junction_max_quantile', 'splice_junction_mean_quantile',
        'splice_site_max_raw', 'splice_site_mean_raw',
        'splice_site_max_quantile', 'splice_site_mean_quantile',
        'n_predictions',
    ]

    # Keep only columns that actually exist (defensive against future schema changes)
    cols = [c for c in column_order if c in summary_df.columns]
    # Append any extra columns (so nothing gets silently dropped)
    extras = [c for c in summary_df.columns if c not in cols]
    output_df = summary_df[cols + extras].copy()

    # Human-readable header for the documented columns
    rename_map = {
        'variant_id': 'Variant ID',
        'short_variant': 'Short Variant',
        'chrom': 'Chromosome',
        'position': 'Position',
        'ref': 'Ref',
        'alt': 'Alt',
        'change': 'Change',
        'impact_class': 'Impact Classification',
        'overall_max_raw': 'Overall Max Raw',
        'overall_mean_raw': 'Overall Mean Raw',
        'overall_max_quantile': 'Overall Max Quantile',
        'overall_mean_quantile': 'Overall Mean Quantile',
        'splice_junction_max_raw': 'Splice Junction Max Raw',
        'splice_junction_mean_raw': 'Splice Junction Mean Raw',
        'splice_junction_max_quantile': 'Splice Junction Max Quantile',
        'splice_junction_mean_quantile': 'Splice Junction Mean Quantile',
        'splice_site_max_raw': 'Splice Site Max Raw',
        'splice_site_mean_raw': 'Splice Site Mean Raw',
        'splice_site_max_quantile': 'Splice Site Max Quantile',
        'splice_site_mean_quantile': 'Splice Site Mean Quantile',
        'n_predictions': 'N Predictions',
    }
    output_df = output_df.rename(columns=rename_map)

    # Round all numeric (float) columns to 6 decimals for readability
    float_cols = output_df.select_dtypes(include='float').columns
    output_df[float_cols] = output_df[float_cols].round(6)

    output_path = os.path.join(output_dir, 'ism_variant_summary.csv')
    output_df.to_csv(output_path, index=False)
    print(f"Saved: ism_variant_summary.csv ({len(output_df):,} variants, {len(output_df.columns)} columns)")

    return output_df

In [9]:
def resolve_intron_input(intron_dir: Path):
    """
    Return the path to load score data for an intron, supporting both layouts:
      * a ``csv/`` subfolder (per-variant files OR a single aggregated file), or
      * an aggregated score file placed directly inside the intron folder.
    Returns None if no score files are found.
    """
    intron_dir = Path(intron_dir)
    csv_dir = intron_dir / 'csv'
    if csv_dir.is_dir() and find_score_files(csv_dir):
        return csv_dir
    if find_score_files(intron_dir):
        return intron_dir
    return None


def discover_intron_dirs(input_root: str, intron_filter=None) -> list:
    """Return sorted intron_* directories that contain loadable score files
    (either in a csv/ subfolder or as an aggregated file in the folder)."""
    root = Path(input_root)
    if not root.is_dir():
        raise FileNotFoundError(f"Input root not found: {root}")

    intron_dirs = []
    for d in sorted(root.iterdir()):
        if not d.is_dir() or not d.name.startswith('intron_'):
            continue
        if resolve_intron_input(d) is None:
            continue
        if intron_filter is not None and d.name not in intron_filter:
            continue
        intron_dirs.append(d)

    if not intron_dirs:
        raise FileNotFoundError(f"No intron_* folders with score files found under {root}")

    def _intron_num(name):
        try:
            return int(name.split('_', 1)[1])
        except (IndexError, ValueError):
            return 9999

    intron_dirs.sort(key=lambda p: _intron_num(p.name))
    return intron_dirs


def intron_number_from_name(name: str) -> int:
    return int(name.split('_', 1)[1])

In [10]:
from scipy import stats


def merge_intervals(df, gap=0):
    """Merge contiguous significant sliding windows into hotspot intervals."""
    if df.empty:
        return df
    df = df.sort_values('window_start').reset_index(drop=True)
    merged = []
    cur_s, cur_e = df.loc[0, 'window_start'], df.loc[0, 'window_end']
    cur_q = df.loc[0, 'binom_q_BH']
    for _, row in df.iloc[1:].iterrows():
        if row['window_start'] <= cur_e + 1 + gap:
            cur_e = max(cur_e, row['window_end'])
            cur_q = min(cur_q, row['binom_q_BH'])
        else:
            merged.append((cur_s, cur_e, cur_q))
            cur_s, cur_e, cur_q = row['window_start'], row['window_end'], row['binom_q_BH']
    merged.append((cur_s, cur_e, cur_q))
    return pd.DataFrame(merged, columns=['hotspot_start', 'hotspot_end', 'min_BH_q'])


def run_spatial_hotspot_analysis(
    summary_df: pd.DataFrame,
    output_dir: str,
    window: int = 15,
    step: int = 1,
    fdr: float = 0.05,
    end_trim_nt: int = 100,
    save_plots: bool = True,
    show_plots: bool = False,
):
    """Spatial clustering + sliding-window hotspot enrichment (intron-6 logic)."""
    hi_pos = summary_df.loc[summary_df['impact_class'] == 'High Impact', 'position'].to_numpy()
    mod_pos = summary_df.loc[summary_df['impact_class'] == 'Moderate Impact', 'position'].to_numpy()
    hm_pos = np.sort(np.concatenate([hi_pos, mod_pos]))

    region_start = int(summary_df['position'].min())
    region_end = int(summary_df['position'].max())
    region_len = region_end - region_start + 1

    interior_start = region_start + end_trim_nt
    interior_end = region_end - end_trim_nt
    if interior_start <= interior_end:
        interior_df = summary_df[summary_df['position'].between(interior_start, interior_end)]
        n_high_interior = int((interior_df['impact_class'] == 'High Impact').sum())
        n_moderate_interior = int((interior_df['impact_class'] == 'Moderate Impact').sum())
        n_variants_interior = int(len(interior_df))
    else:
        n_high_interior = 0
        n_moderate_interior = 0
        n_variants_interior = 0
    n_HM_interior = n_high_interior + n_moderate_interior
    fraction_HM_interior = (
        n_HM_interior / n_variants_interior if n_variants_interior else 0.0
    )

    def mean_nn_distance(positions):
        sp = np.sort(np.unique(positions))
        return float(np.mean(np.diff(sp))) if len(sp) > 1 else np.nan

    all_unique_pos = np.sort(summary_df['position'].unique())

    def perm_pvalue(observed_pos, n_perm=10_000, seed=0):
        obs = mean_nn_distance(observed_pos)
        n_unique = pd.unique(observed_pos).size
        if n_unique <= 1:
            return obs, np.nan, np.nan
        # The mean nearest-neighbour distance of a sorted set telescopes to
        # (max - min) / (n_unique - 1). For a fixed sample size the divisor is
        # constant, so each permutation only needs the span of a random
        # size-n_unique subset drawn without replacement. Vectorised in
        # memory-bounded chunks instead of a per-permutation Python loop.
        M = all_unique_pos.size
        denom = n_unique - 1
        rng = np.random.default_rng(seed)
        perm = np.empty(n_perm)
        rows_per_chunk = max(1, int(20_000_000 // max(M, 1)))
        filled = 0
        while filled < n_perm:
            c = min(rows_per_chunk, n_perm - filled)
            keys = rng.random((c, M))
            idx = np.argpartition(keys, n_unique - 1, axis=1)[:, :n_unique]
            sampled = all_unique_pos[idx]
            perm[filled:filled + c] = (sampled.max(axis=1) - sampled.min(axis=1)) / denom
            filled += c
        p = (np.sum(perm <= obs) + 1) / (n_perm + 1)
        return obs, float(perm.mean()), p

    uniform_cdf = stats.uniform(loc=region_start - 0.5, scale=region_len).cdf
    ks_hm = stats.kstest(hm_pos, uniform_cdf) if len(hm_pos) else None
    ks_hi = stats.kstest(hi_pos, uniform_cdf) if len(hi_pos) else None
    ks_mo = stats.kstest(mod_pos, uniform_cdf) if len(mod_pos) else None
    obs_hm, exp_hm, p_perm_hm = perm_pvalue(hm_pos, seed=1) if len(hm_pos) else (np.nan, np.nan, np.nan)
    obs_hi, exp_hi, p_perm_hi = perm_pvalue(hi_pos, seed=2) if len(hi_pos) else (np.nan, np.nan, np.nan)

    bg_frac = len(hm_pos) / len(summary_df) if len(summary_df) else 0.0
    window_starts = np.arange(region_start, region_end - window + 2, step)
    # Vectorised sliding-window counts via per-position tallies + prefix sums
    # (O(L + N) instead of the previous O(L * N) per-window DataFrame masking).
    offsets = (summary_df['position'].to_numpy() - region_start).astype(np.int64)
    is_hm = summary_df['impact_class'].isin(['High Impact', 'Moderate Impact']).to_numpy()
    total_per_pos = np.bincount(offsets, minlength=region_len)
    hm_per_pos = np.bincount(offsets, weights=is_hm.astype(float), minlength=region_len)
    cum_total = np.concatenate(([0], np.cumsum(total_per_pos)))
    cum_hm = np.concatenate(([0.0], np.cumsum(hm_per_pos)))
    start_off = window_starts - region_start
    end_off = start_off + window
    n_in_win = (cum_total[end_off] - cum_total[start_off]).astype(int)
    k_in_win = np.rint(cum_hm[end_off] - cum_hm[start_off]).astype(int)

    # Vectorised one-sided binomial test: P(X >= k) = binom.sf(k - 1, n, p),
    # identical to stats.binomtest(..., alternative='greater').
    binom_p = stats.binom.sf(k_in_win - 1, n_in_win, bg_frac)
    binom_p = np.where(n_in_win > 0, binom_p, 1.0)
    order = np.argsort(binom_p)
    ranked = binom_p[order]
    m = len(ranked)
    adj = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1] if m else np.array([])
    binom_q = np.empty_like(binom_p)
    if m:
        binom_q[order] = np.clip(adj, 0, 1)

    window_centers = window_starts + window // 2
    window_frac = np.divide(k_in_win, n_in_win, out=np.zeros_like(k_in_win, dtype=float), where=n_in_win > 0)

    hotspots = pd.DataFrame({
        'window_start': window_starts,
        'window_end': window_starts + window - 1,
        'n_total': n_in_win,
        'n_HM': k_in_win,
        'fraction_HM': window_frac,
        'binom_p': binom_p,
        'binom_q_BH': binom_q,
    })
    sig_windows = hotspots[hotspots['binom_q_BH'] < fdr].copy()
    hotspot_intervals = merge_intervals(sig_windows)

    if not hotspot_intervals.empty:
        hotspot_intervals['length_nt'] = (
            hotspot_intervals['hotspot_end'] - hotspot_intervals['hotspot_start'] + 1
        )
        hotspot_intervals['n_HM_in_hotspot'] = [
            int(summary_df[
                (summary_df['position'].between(s, e)) &
                (summary_df['impact_class'].isin(['High Impact', 'Moderate Impact']))
            ].shape[0])
            for s, e in zip(hotspot_intervals['hotspot_start'], hotspot_intervals['hotspot_end'])
        ]
        hotspot_intervals['n_total_in_hotspot'] = [
            int(summary_df[summary_df['position'].between(s, e)].shape[0])
            for s, e in zip(hotspot_intervals['hotspot_start'], hotspot_intervals['hotspot_end'])
        ]
        hotspot_intervals['fraction_HM'] = (
            hotspot_intervals['n_HM_in_hotspot'] / hotspot_intervals['n_total_in_hotspot']
        ).round(3)

    if save_plots:
        color_hi = '#b30000'
        color_mod = '#fdae6b'
        color_bg = '#bdbdbd'
        fig, axes = plt.subplots(
            2, 1, figsize=(15, 3.8), sharex=True,
            gridspec_kw={'height_ratios': [1.0, 1.3]},
        )
        ax = axes[0]
        # Adaptive bin width: scale with region length so bars stay visible
        # for very long introns (target ~target_bins bars across the panel).
        target_bins = 400
        bin_width = max(10, int(round(region_len / target_bins / 10.0)) * 10)
        bin_edges = np.arange(region_start, region_end + bin_width + 1, bin_width)
        # Thin/no edge lines when there are many bars, otherwise white separators.
        n_bins = len(bin_edges) - 1
        bar_edgecolor = 'white' if n_bins <= 200 else 'none'
        bar_lw = 0.6 if n_bins <= 200 else 0.0
        ax.hist([hi_pos, mod_pos], bins=bin_edges, stacked=True,
                color=[color_hi, color_mod],
                label=['High Impact', 'Moderate Impact'],
                edgecolor=bar_edgecolor, linewidth=bar_lw)
        ax.set_ylabel(f'# H/M variants\nper {bin_width} nt window', fontsize=10)
        ax.tick_params(axis='both', labelsize=9)
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)

        ax = axes[1]
        ax.plot(window_centers, window_frac, color='black', linewidth=1.4,
                label=f'Fraction H+M ({window}-nt window)')
        ax.axhline(bg_frac, color=color_bg, linestyle='--', linewidth=1,
                   label=f'Background H+M fraction = {bg_frac:.1%}')
        sig_mask = (binom_q < fdr)
        ax.fill_between(window_centers, 0, window_frac, where=sig_mask,
                        color=color_hi, alpha=0.35, step='mid',
                        label=f'Enriched window (BH q<{fdr})')
        ax.set_xlim(region_start - 1, region_end + 1)
        ax.set_ylim(0, max(0.6, np.nanmax(window_frac) * 1.05) if len(window_frac) else 0.6)
        chrom = summary_df['chrom'].iloc[0].lstrip('chr')
        ax.set_xlabel(f'chr{chrom} position (hg38)', fontsize=10)
        ax.set_ylabel('Fraction of variants\nclassified H or M', fontsize=10)
        ax.tick_params(axis='both', labelsize=9)
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout(rect=[0, 0, 0.8, 1])
        plt.savefig(os.path.join(output_dir, 'spatial_clustering_HM_variants.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(output_dir, 'spatial_clustering_HM_variants.svg'), bbox_inches='tight')
        if show_plots:
            plt.show()
        else:
            plt.close(fig)

    if save_plots and not hotspot_intervals.empty:
        hotspot_intervals.to_csv(os.path.join(output_dir, 'spatial_hotspots_HM_variants.csv'), index=False)

    stats_out = {
        'region_start': region_start,
        'region_end': region_end,
        'region_len_nt': region_len,
        'n_variants': len(summary_df),
        'n_high': int(len(hi_pos)),
        'n_moderate': int(len(mod_pos)),
        'n_HM': int(len(hm_pos)),
        'fraction_HM': bg_frac,
        'n_high_interior': n_high_interior,
        'n_moderate_interior': n_moderate_interior,
        'n_HM_interior': n_HM_interior,
        'fraction_HM_interior': fraction_HM_interior,
        'n_variants_interior': n_variants_interior,
        'ks_HM_D': ks_hm.statistic if ks_hm else np.nan,
        'ks_HM_p': ks_hm.pvalue if ks_hm else np.nan,
        'ks_high_D': ks_hi.statistic if ks_hi else np.nan,
        'ks_high_p': ks_hi.pvalue if ks_hi else np.nan,
        'mean_nn_HM': obs_hm,
        'mean_nn_HM_perm_p': p_perm_hm,
        'n_sig_windows': int(len(sig_windows)),
        'n_hotspot_intervals': int(len(hotspot_intervals)),
        'max_hotspot_fraction_HM': float(hotspot_intervals['fraction_HM'].max()) if not hotspot_intervals.empty else np.nan,
        'max_window_fraction_HM': float(window_frac.max()) if len(window_frac) else np.nan,
        'max_splice_site_quantile': float(summary_df['splice_site_max_quantile'].max()),
    }
    return stats_out, hotspot_intervals, hotspots


In [11]:
def run_intron_analysis(
    intron_dir: Path,
    output_root: str,
    threshold_high=0.999,
    threshold_moderate=0.99,
    threshold_low=0.95,
    hotspot_window=15,
    hotspot_step=1,
    hotspot_fdr=0.05,
    end_trim_nt=100,
    generate_ism_heatmap=True,
    generate_scores_heatmap=True,
    show_plots=False,
):
    """Run full ISM + hotspot workflow for one intron folder."""
    intron_name = intron_dir.name
    intron_num = intron_number_from_name(intron_name)
    csv_path = resolve_intron_input(intron_dir)
    if csv_path is None:
        raise FileNotFoundError(f'No score files found for {intron_name}')
    out_dir = os.path.join(output_root, intron_name)
    os.makedirs(out_dir, exist_ok=True)

    print('\n' + '=' * 70)
    print(f'Processing {intron_name} (intron {intron_num})')
    print('=' * 70)

    combined_df = load_alphagenome_data(str(csv_path))
    summary_df = create_variant_summary(
        combined_df,
        threshold_high=threshold_high,
        threshold_moderate=threshold_moderate,
        threshold_low=threshold_low,
    )
    print_summary_statistics(summary_df)
    save_summary_table(summary_df, out_dir)

    if generate_ism_heatmap:
        plot_ism_heatmap(summary_df, out_dir, top_n=None, show=show_plots)

    if generate_scores_heatmap:
        plot_scores_heatmap(combined_df, summary_df, out_dir, top_n=TOP_N_VARIANTS, show=show_plots)

    hotspot_stats, hotspot_intervals, _ = run_spatial_hotspot_analysis(
        summary_df,
        out_dir,
        window=hotspot_window,
        step=hotspot_step,
        fdr=hotspot_fdr,
        end_trim_nt=end_trim_nt,
        save_plots=True,
        show_plots=show_plots,
    )

    row = {
        'intron': intron_name,
        'intron_number': intron_num,
        **hotspot_stats,
    }
    if not hotspot_intervals.empty:
        best = hotspot_intervals.sort_values('fraction_HM', ascending=False).iloc[0]
        row['top_hotspot_start'] = int(best['hotspot_start'])
        row['top_hotspot_end'] = int(best['hotspot_end'])
        row['top_hotspot_fraction_HM'] = float(best['fraction_HM'])
        row['top_hotspot_length_nt'] = int(best['length_nt'])
    else:
        row['top_hotspot_start'] = np.nan
        row['top_hotspot_end'] = np.nan
        row['top_hotspot_fraction_HM'] = np.nan
        row['top_hotspot_length_nt'] = np.nan

    print(f"Saved outputs to {out_dir}")
    return row, summary_df, hotspot_intervals


In [12]:
def plot_cohort_overview(cohort_df: pd.DataFrame, output_dir: str):
    """Cross-intron summary plots for hotspot comparison."""
    df = cohort_df.sort_values('intron_number').copy()
    df['intron_label'] = df['intron_number'].astype(str)
    x = np.arange(len(df))

    # --- 1) H/M burden per intron (all variants) ---
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(x, df['n_high'], color='#b30000', label='High Impact')
    ax.bar(x, df['n_moderate'], bottom=df['n_high'], color='#fdae6b', label='Moderate Impact')
    ax.set_ylabel('# H/M variants')
    ax.set_xlabel(f'{GENE_NAME} intron number')
    ax.set_xticks(x)
    ax.set_xticklabels(df['intron_label'], rotation=90, fontsize=8)
    ax.legend(frameon=False, loc='upper right')
    ax.set_title(f'{GENE_NAME} intronic ISM — hotspot signal across introns', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'cohort_overview_HM_burden.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_dir, 'cohort_overview_HM_burden.svg'), bbox_inches='tight')
    plt.show()

    # --- 2) H/M burden per intron excluding first/last trim nt ---
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(x, df['n_high_interior'], color='#b30000', label='High Impact')
    ax.bar(
        x,
        df['n_moderate_interior'],
        bottom=df['n_high_interior'],
        color='#fdae6b',
        label='Moderate Impact',
    )
    ax.set_ylabel('# H/M variants')
    ax.set_xlabel(f'{GENE_NAME} intron number')
    ax.set_xticks(x)
    ax.set_xticklabels(df['intron_label'], rotation=90, fontsize=8)
    ax.legend(frameon=False, loc='upper right')
    ax.set_title(
        f'{GENE_NAME} H/M variants excluding first/last {INTRON_END_TRIM_NT} nt per intron',
        fontweight='bold',
    )
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'cohort_overview_HM_burden_interior.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_dir, 'cohort_overview_HM_burden_interior.svg'), bbox_inches='tight')
    plt.show()

    print('Saved cohort overview plots to', output_dir)


---
## 8. Run Per-Intron Analysis

Loop over every `intron_*` folder under `INPUT_ROOT`, apply the same
pathogenicity cutoffs and spatial hotspot tests as the intron-6 workflow,
and write per-intron plots/tables to `OUTPUT_DIR/intron_*/`.

### 8.1 Discover intron folders

In [13]:
intron_dirs = discover_intron_dirs(INPUT_ROOT, INTRON_FILTER)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Input root : {INPUT_ROOT}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Introns to process: {len(intron_dirs)}')
for d in intron_dirs:
    src = resolve_intron_input(d)
    files = find_score_files(src) if src is not None else []
    if len(files) == 1:
        print(f'  {d.name}: 1 aggregated CSV ({Path(files[0]).name})')
    else:
        print(f'  {d.name}: {len(files):,} CSV files')

Input root : ../01_alphagenome_analysis/alphagenome_ISM_COL4A5/
Output dir : ./plots_alphagenome_ISM_COL4A5_all_introns/
Introns to process: 52
  intron_1: 1 aggregated CSV (alphagenome_scores_intron_1.csv)
  intron_2: 1 aggregated CSV (alphagenome_scores_intron_2.csv)
  intron_3: 1 aggregated CSV (alphagenome_scores_intron_3.csv)
  intron_4: 1 aggregated CSV (alphagenome_scores_intron_4.csv)
  intron_5: 256 CSV files
  intron_6: 7,774 CSV files
  intron_7: 1,033 CSV files
  intron_8: 5,209 CSV files
  intron_9: 6,766 CSV files
  intron_10: 5,938 CSV files
  intron_11: 271 CSV files
  intron_12: 514 CSV files
  intron_13: 6,448 CSV files
  intron_14: 286 CSV files
  intron_15: 733 CSV files
  intron_16: 5,569 CSV files
  intron_17: 4,639 CSV files
  intron_18: 6,268 CSV files
  intron_19: 1 aggregated CSV (alphagenome_scores_intron_19.csv)
  intron_20: 988 CSV files
  intron_21: 11,593 CSV files
  intron_22: 4,189 CSV files
  intron_23: 925 CSV files
  intron_24: 3,400 CSV files
  intr

### 8.2 Process each intron

In [ ]:
cohort_rows = []
all_hotspot_intervals = []

for intron_dir in intron_dirs:
    row, _, hotspot_intervals = run_intron_analysis(
        intron_dir,
        OUTPUT_DIR,
        threshold_high=THRESHOLD_HIGH,
        threshold_moderate=THRESHOLD_MODERATE,
        threshold_low=THRESHOLD_LOW,
        hotspot_window=HOTSPOT_WINDOW,
        hotspot_step=HOTSPOT_STEP,
        hotspot_fdr=HOTSPOT_FDR,
        end_trim_nt=INTRON_END_TRIM_NT,
        generate_ism_heatmap=GENERATE_ISM_HEATMAP,
        generate_scores_heatmap=GENERATE_SCORES_HEATMAP,
        show_plots=False,
    )
    cohort_rows.append(row)
    if not hotspot_intervals.empty:
        hi = hotspot_intervals.copy()
        hi['intron'] = intron_dir.name
        hi['intron_number'] = intron_number_from_name(intron_dir.name)
        all_hotspot_intervals.append(hi)

cohort_df = pd.DataFrame(cohort_rows).sort_values('intron_number').reset_index(drop=True)
# Tag the in-memory results with the gene they belong to so downstream
# cells don't reuse stale data after GENE_NAME is changed.
cohort_gene = GENE_NAME
cohort_csv = os.path.join(OUTPUT_DIR, 'cohort_intron_summary.csv')
cohort_df.to_csv(cohort_csv, index=False)
print(f'\nSaved cohort summary: {cohort_csv}')
display(cohort_df)

### 8.3 Cohort-level hotspot overview

Compare H/M variant burden (full intron and interior excluding splice-adjacent ends),
and significant hotspot intervals across all introns of the gene.

In [ ]:
cohort_csv = os.path.join(OUTPUT_DIR, 'cohort_intron_summary.csv')
_have_current_cohort = (
    'cohort_df' in globals()
    and cohort_df is not None
    and globals().get('cohort_gene') == GENE_NAME
)
if not _have_current_cohort:
    if os.path.exists(cohort_csv):
        cohort_df = pd.read_csv(cohort_csv)
        cohort_gene = GENE_NAME
        print(f'Loaded cohort summary from {cohort_csv}')
    else:
        raise FileNotFoundError(
            f'{cohort_csv} not found. Run section 8.2 to generate the cohort summary for {GENE_NAME} first.'
        )

plot_cohort_overview(cohort_df, OUTPUT_DIR)

# Load per-intron hotspot intervals for the current gene from memory or disk.
all_hotspots_csv = os.path.join(OUTPUT_DIR, 'all_introns_hotspot_intervals.csv')
all_hotspots_df = None
if _have_current_cohort and 'all_hotspot_intervals' in globals() and all_hotspot_intervals:
    all_hotspots_df = pd.concat(all_hotspot_intervals, ignore_index=True)
    all_hotspots_df.to_csv(all_hotspots_csv, index=False)
    print(f'Saved: {all_hotspots_csv}')
elif os.path.exists(all_hotspots_csv):
    all_hotspots_df = pd.read_csv(all_hotspots_csv)
    print(f'Loaded: {all_hotspots_csv}')

if all_hotspots_df is not None and not all_hotspots_df.empty:
    display(all_hotspots_df.sort_values(['intron_number', 'fraction_HM'], ascending=[True, False]).head(20))
else:
    print('No significant hotspot intervals detected in any intron at the chosen FDR.')

### 8.4 Deep-intronic hotspot concentration (clustering vs spread)

A companion to `cohort_overview_HM_burden_interior.png`. Raw H/M counts favour long
introns (e.g. intron 1) that can accumulate many moderate variants scattered over
kilobases of deep intronic sequence. This section instead measures **how tightly the
deep-intronic H/M variants are clustered into significant hotspots**, independent of
intron length.

For each intron we compute the **hotspot concentration**:

concentration = deep-intronic H/M variants inside significant hotspot intervals divided to deep-intronic H/M variants

where hotspots are the BH `q < HOTSPOT_FDR` sliding-window intervals whose centre lies
in the interior region (excluding the first/last `INTRON_END_TRIM_NT` nt). A value near
**1** means the H/M signal is packed into a few dense hotspots (a strong, focal hotspot
intron); a value near **0** means variants are spread thinly across the intron. Bars are
coloured by the **number** of H/M variants sitting in those hotspots, so tall + dark bars
are the strongest, most abundant deep-intronic hotspots.

Saved as `cohort_hotspot_concentration.png/svg`.

In [ ]:
import matplotlib as mpl

# Control which introns APPEAR in the figure. Both only affect the plot;
# the saved CSV always contains every intron (nothing is deleted).
#   KEEP_INTRONS: show only these, e.g. [1, 6, 24, 47]; None = show all.
#   HIDE_INTRONS: omit bars for these introns but keep their x-axis labels.
KEEP_INTRONS = None
HIDE_INTRONS = [2, 18, 21, 25, 36, 42]
# Order bars by concentration (descending) instead of by intron number.
SORT_BY_CONCENTRATION = False
# Concentration mode:
# - 'sum_hotspots': sum H/M across all significant interior hotspots
# - 'top_hotspot' : use one representative interior hotspot per intron (max n_HM_in_hotspot)
CONCENTRATION_MODE = 'top_hotspot'

# Self-contained loading for the CURRENT gene (mirrors 8.3)
cohort_csv = os.path.join(OUTPUT_DIR, 'cohort_intron_summary.csv')
_have_current_cohort = (
    'cohort_df' in globals() and cohort_df is not None
    and globals().get('cohort_gene') == GENE_NAME
)
if not _have_current_cohort:
    if os.path.exists(cohort_csv):
        cohort_df = pd.read_csv(cohort_csv)
        cohort_gene = GENE_NAME
        print(f'Loaded cohort summary from {cohort_csv}')
    else:
        raise FileNotFoundError(
            f'{cohort_csv} not found. Run section 8.2 for {GENE_NAME} first.'
        )

all_hotspots_csv = os.path.join(OUTPUT_DIR, 'all_introns_hotspot_intervals.csv')
if _have_current_cohort and 'all_hotspots_df' in globals() and all_hotspots_df is not None:
    _hs = all_hotspots_df.copy()
elif os.path.exists(all_hotspots_csv):
    _hs = pd.read_csv(all_hotspots_csv)
    print(f'Loaded hotspot intervals from {all_hotspots_csv}')
else:
    _hs = pd.DataFrame(columns=['hotspot_start', 'hotspot_end', 'n_HM_in_hotspot', 'intron_number'])

# Restrict hotspots to the deep-intronic interior and tally H/M per intron
conc = cohort_df[['intron_number', 'region_start', 'region_end', 'n_HM_interior']].copy()
conc = conc.sort_values('intron_number').reset_index(drop=True)

n_hm_in_hotspots = pd.Series(0.0, index=conc['intron_number'])
if not _hs.empty:
    hs = _hs.merge(
        conc[['intron_number', 'region_start', 'region_end']],
        on='intron_number', how='left',
    )
    hs['center'] = (hs['hotspot_start'] + hs['hotspot_end']) / 2.0
    interior_lo = hs['region_start'] + INTRON_END_TRIM_NT
    interior_hi = hs['region_end'] - INTRON_END_TRIM_NT
    interior_hs = hs[(hs['center'] >= interior_lo) & (hs['center'] <= interior_hi)]
    if not interior_hs.empty:
        if CONCENTRATION_MODE == 'top_hotspot':
            # Use one representative hotspot per intron (largest H/M count in hotspot).
            top = (
                interior_hs
                .sort_values(['intron_number', 'n_HM_in_hotspot', 'hotspot_start'], ascending=[True, False, True])
                .drop_duplicates(subset=['intron_number'], keep='first')
            )
            grp = top.set_index('intron_number')['n_HM_in_hotspot']
            n_hm_in_hotspots.loc[grp.index] = grp.values
        else:
            grp = interior_hs.groupby('intron_number')['n_HM_in_hotspot'].sum()
            n_hm_in_hotspots.loc[grp.index] = grp.values

conc['n_HM_in_interior_hotspots'] = (
    conc['intron_number'].map(n_hm_in_hotspots).fillna(0).astype(int)
)
conc['concentration'] = np.where(
    conc['n_HM_interior'] > 0,
    conc['n_HM_in_interior_hotspots'] / conc['n_HM_interior'],
    0.0,
).clip(0, 1)

# Optionally choose which introns appear in the figure (plot only)
conc_plot = conc.copy()
suffix = ''
if KEEP_INTRONS is not None:
    missing = sorted(set(KEEP_INTRONS) - set(conc['intron_number']))
    if missing:
        print(f'Warning: KEEP_INTRONS not found and skipped: {missing}')
    conc_plot = conc_plot[conc_plot['intron_number'].isin(KEEP_INTRONS)]
    suffix = '_subset'
if SORT_BY_CONCENTRATION:
    conc_plot = conc_plot.sort_values(
        ['concentration', 'n_HM_in_interior_hotspots'], ascending=False
    )
else:
    conc_plot = conc_plot.sort_values('intron_number')
conc_plot = conc_plot.reset_index(drop=True)
if conc_plot.empty:
    raise ValueError('No introns left to plot after applying KEEP_INTRONS.')

hide_set = set(HIDE_INTRONS or [])
if hide_set:
    suffix = suffix or '_subset'
if CONCENTRATION_MODE == 'top_hotspot':
    suffix = f'{suffix}_tophotspot' if suffix else '_tophotspot'

# Plot: bar height = concentration, colour = # H/M variants in representative hotspots
x = np.arange(len(conc_plot))
show_mask = ~conc_plot['intron_number'].isin(hide_set).to_numpy()
counts = conc_plot['n_HM_in_interior_hotspots'].to_numpy()
norm = mpl.colors.Normalize(vmin=0, vmax=max(1, int(counts.max())))
cmap = plt.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(
    x[show_mask],
    conc_plot.loc[show_mask, 'concentration'].to_numpy(),
    color=cmap(norm(counts[show_mask])),
    edgecolor='black',
    linewidth=0.3,
)
ax.set_xticks(x)
ax.set_xticklabels(conc_plot['intron_number'].astype(str), rotation=90, fontsize=8)
ax.set_xlabel(f'{GENE_NAME} intron number', fontsize=10)
ax.set_ylabel('Fraction of deep-intronic H/M\nvariants in representative hotspot', fontsize=10)
ax.set_ylim(0, 1.0)
ax.tick_params(axis='both', labelsize=9)
ax.spines[['top', 'right']].set_visible(False)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label('# H/M variants in\nrepresentative hotspot', fontsize=9)
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'cohort_hotspot_concentration{suffix}.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, f'cohort_hotspot_concentration{suffix}.svg'), bbox_inches='tight')
plt.show()

conc_csv = os.path.join(OUTPUT_DIR, f'cohort_hotspot_concentration{suffix}.csv')
conc.to_csv(conc_csv, index=False)
print(f'Saved: {conc_csv}')
display(
    conc_plot.sort_values(['concentration', 'n_HM_in_interior_hotspots'], ascending=False)[
        ['intron_number', 'n_HM_interior', 'n_HM_in_interior_hotspots', 'concentration']
    ].head(15)
)

### 8.5 Focused ISM heatmaps for significant hotspot regions

**Self-contained** — set `FOCUS_INTRON` in the cell below and run it. It loads the
already-generated per-intron results from disk (`ism_variant_summary.csv` +
`spatial_hotspots_HM_variants.csv`), so there is **no need to re-run 8.2** or
reprocess any data.

For the chosen intron it takes every **significant** sliding-window hotspot
(merged interval with BH q < `HOTSPOT_FDR`), pads it by `HOTSPOT_PAD_NT`
nucleotides up- and downstream, and renders a focused saturation-mutagenesis
**ISM heatmap** (same style as intron 6 / intron 47), exporting a **PNG** (+ SVG)
per hotspot under `<intron>/hotspot_heatmaps/`.

In [ ]:
# Focused ISM heatmaps for significant (BH q < FDR) hotspot regions.
FOCUS_INTRON = 'intron_25'   # choose the intron to analyse, e.g. 'intron_6', 'intron_47'
HOTSPOT_PAD_NT = 10         # nt added upstream & downstream of each hotspot interval


def _load_intron_results(intron_name, results_root=OUTPUT_DIR):
    """Load pre-computed variant summary + hotspot intervals for one intron from disk."""
    intron_dir = os.path.join(results_root, intron_name)
    summary_csv = os.path.join(intron_dir, 'ism_variant_summary.csv')
    hotspot_csv = os.path.join(intron_dir, 'spatial_hotspots_HM_variants.csv')
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(
            f'No ism_variant_summary.csv for {intron_name} at {summary_csv}. '
            f'Run the cohort batch first.'
        )
    summary_df = pd.read_csv(summary_csv)
    summary_df.columns = [c.strip() for c in summary_df.columns]
    summary_df = summary_df.rename(columns={
        'Position': 'position',
        'Alt': 'alt',
        'Chromosome': 'chrom',
        'Impact Classification': 'impact_class',
        'Splice Site Max Quantile': 'splice_site_max_quantile',
    })
    if os.path.exists(hotspot_csv):
        hotspot_intervals = pd.read_csv(hotspot_csv)
    else:
        hotspot_intervals = pd.DataFrame()
        print(f'(No spatial_hotspots_HM_variants.csv for {intron_name} - '
            f'no significant hotspots were found for this intron.)')
    return summary_df, hotspot_intervals


def plot_hotspot_region_heatmaps(summary_df, hotspot_intervals, output_dir,
                                pad_nt=HOTSPOT_PAD_NT, fdr=HOTSPOT_FDR, show=False):
    """Render & export a focused ISM heatmap PNG/SVG for each significant hotspot.

    A hotspot is significant when its merged minimum BH q-value < ``fdr``. The
    plotted window is padded by ``pad_nt`` nt on each side so the flanks of the
    interesting region are visible. Each hotspot is written to its own
    sub-folder ``hotspot_<chrom>_<start>_<end>/ism_heatmap.png``.
    """
    if hotspot_intervals is None or len(hotspot_intervals) == 0:
        print('No hotspot intervals available - nothing to plot.')
        return []

    sig = hotspot_intervals.copy()
    if 'min_BH_q' in sig.columns:
        sig = sig[sig['min_BH_q'] < fdr]
    if sig.empty:
        print(f'No hotspots with BH q < {fdr}.')
        return []

    os.makedirs(output_dir, exist_ok=True)
    chrom = str(summary_df['chrom'].iloc[0])
    saved = []
    for _, hs in sig.sort_values('hotspot_start').iterrows():
        h_start, h_end = int(hs['hotspot_start']), int(hs['hotspot_end'])
        win_start, win_end = h_start - pad_nt, h_end + pad_nt
        sub = summary_df[summary_df['position'].between(win_start, win_end)].copy()
        if sub.empty:
            continue
        hs_dir = os.path.join(output_dir, f'hotspot_{chrom}_{win_start}_{win_end}')
        os.makedirs(hs_dir, exist_ok=True)
        q = float(hs.get('min_BH_q', np.nan))
        n_hm = int(sub['impact_class'].isin(['High Impact', 'Moderate Impact']).sum())
        print(f'{chrom}:{h_start}-{h_end} (+/-{pad_nt} nt -> {win_start}-{win_end}, '
            f'{win_end - win_start + 1} nt): {len(sub)} variants, {n_hm} H/M, q={q:.2e}')
        plot_ism_heatmap(sub, output_dir=hs_dir, top_n=None, show=show)
        saved.append(os.path.join(hs_dir, 'ism_heatmap.png'))
    print(f'\nSaved {len(saved)} hotspot heatmap PNG(s) under: {output_dir}')
    return saved


# --- Run focused analysis for FOCUS_INTRON using the pre-generated results ---
_summary_focus, _hotspots_focus = _load_intron_results(FOCUS_INTRON)
_focus_out = os.path.join(OUTPUT_DIR, FOCUS_INTRON, 'hotspot_heatmaps')
print(f'Focused hotspot ISM heatmaps for {FOCUS_INTRON} '
    f'(+/-{HOTSPOT_PAD_NT} nt, BH q < {HOTSPOT_FDR}):\n')
hotspot_heatmap_pngs = plot_hotspot_region_heatmaps(
    _summary_focus,
    _hotspots_focus,
    _focus_out,
    pad_nt=HOTSPOT_PAD_NT,
    fdr=HOTSPOT_FDR,
    show=True,
)

---
## 9. Interpretation Guide

### Understanding the Results

**Quantile Scores:**
- Scores range from 0 to 1
- Higher scores indicate stronger predicted effects on splicing
- A score of 0.999 means the variant has a stronger effect than 99.9% of all possible variants

**Impact Classification:**
- **High Impact (≥0.999)**: Top 0.1% - Very likely to disrupt splicing
- **Moderate Impact (≥0.99)**: Top 1% - Likely to affect splicing
- **Low Impact (≥0.95)**: Top 5% - May have minor effects
- **No Impact (<0.95)**: Below top 5% - Unlikely to significantly affect splicing

**Key Visualizations:**
1. **ISM Heatmap**: Shows the complete saturation mutagenesis landscape. Hotspots (red regions) indicate positions where mutations are most likely to disrupt splicing.
2. **Score Distribution**: Shows how variants are distributed across impact classes.
3. **Genomic Track**: Highlights positions with the highest impact scores.
4. **Position Impact Summary**: Shows how many high/moderate/low impact variants exist at each position.